# MMIP Week7：多模態多輪客服、四種分流實驗與整合介面
先在 Colab 的「執行階段 → 變更執行階段類型」選 GPU。本筆記本已內含完整作業檔，無需另外上傳壓縮包。依序執行；所有模型訓練與推論都在 Colab，電腦本機只存放作業檔。

In [1]:
from google.colab import files
from pathlib import Path
import base64
import io
import zipfile
payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as archive:
    archive.extractall('/content')
assert Path('/content/MMIP_Week7/requirements.txt').exists()
%cd /content/MMIP_Week7

/content/MMIP_Week7


## 1. 安裝環境並檢查 GPU

In [2]:
%pip -q install -r requirements.txt
import torch
assert torch.cuda.is_available(), '請先將 Colab 執行階段切換成 GPU'
print('GPU:', torch.cuda.get_device_name(0))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
GPU: Tesla T4


In [6]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 2. 產生資料並執行不需 GPU 的控制流程驗證
已有 160 筆訓練樣本、20 筆獨立 Benchmark 與五張合成測試圖片。

In [3]:
!python make_data.py
!python verify.py

Created 160 training examples, 20 held-out cases and 5 synthetic images
test_off_topic_does_not_poison_existing_ticket (__main__.ContractTests.test_off_topic_does_not_poison_existing_ticket) ... ok
test_strict_schema (__main__.ContractTests.test_strict_schema) ... ok
test_text_and_image_injection_are_data (__main__.ContractTests.test_text_and_image_injection_are_data) ... ok
test_ticket_requires_confirmation (__main__.ContractTests.test_ticket_requires_confirmation) ... ok
test_topic_switch_drops_old_context (__main__.ContractTests.test_topic_switch_drops_old_context) ... ok
test_train_and_benchmark_are_distinct (__main__.ContractTests.test_train_and_benchmark_are_distinct) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.006s

OK


## 3. 實驗 B：在 Colab GPU 訓練 LoRA 分流節點

In [7]:
!python train.py --epochs 3

I0000 00:00:1795074657.070529    1381 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795074660.445655    1381 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
`torch_dtype` is deprecated! Use `dtype` instead!
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497
{'loss': 0.4818, 'grad_norm': 1.3709497451782227, 'learning_rate': 0.00019984815164333163, 'epoch': 0.25}
{'loss': 0.1519, 'grad_norm': 1.1827536821365356, 'learning_rate': 0.00019458172417006347, 'epoch': 0.5}
{'loss': 0.046, 'grad_norm': 0.41858983039855957, 'learning_rate': 0.0001821777815225245, 'epoch': 0.75}
{'loss': 0.0168, 'grad_norm': 0.7170713543891907, 'learning_rate': 0.00016357237482099684, 'epoch': 1.0}
{'loss': 0.0045, 'grad_no

## 4. 實驗 A/B/C/D：實際圖片理解與量化比較
下列 USD 1/小時只是統一的計算成本基準，並非 Colab 帳單；若知道實際費率可自行修改。

In [8]:
!python benchmark.py --vision model --gpu-hourly-usd 1.0
from IPython.display import Markdown, display
display(Markdown(Path('artifacts/benchmark_model/results.md').read_text(encoding='utf-8')))

I0000 00:00:1795074774.910251    1913 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795074777.618185    1913 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
preprocessor_config.json: 100% 350/350 [00:00<00:00, 2.03MB/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
tokenizer_config.json: 5.70kB [00:00, 21.5MB/s]
vocab.json: 2.78MB [00:00, 92.5MB/s]
merges.txt: 1.67MB [00:00, 138MB/s]

# Week7 實測結果

圖片來源：Qwen2.5-VL-3B-Instruct 實際觀察
GPU：Tesla T4；估算費率：USD 1.0/小時

| 方法 | 筆數 | 結構正確率 | 分流正確率 | 延遲中位數 | P95 | GPU 等值成本 |
|---|---:|---:|---:|---:|---:|---:|
| A | 20 | 40.0% | 0.0% | 2.169s | 5.500s | $0.01808 |
| B | 20 | 75.0% | 55.0% | 3.317s | 9.531s | $0.02702 |
| C | 20 | 95.0% | 15.0% | 10.360s | 13.457s | $0.06260 |
| D | 20 | 100.0% | 45.0% | 0.002s | 3.400s | $0.00576 |

成本為 GPU 推論時間乘指定費率的估算；D 的 CPU 推論不計 GPU 費用。非 Colab 實際帳單；模型下載與訓練另計。

A = Prompt；B = SFT LoRA；C = 逐 token JSON Schema 約束；D = 字元 n-gram + 邏輯回退。
四種方法使用相同模型可見的對話和圖片觀察；D 是較小的 System One 節點。
結構正確率要求全部欄位、型別與 route enum 正確。分流正確率按預先標註的 route 判斷。
完整逐題輸出與錯誤原因在 all_results.csv。


## 5. 下載訓練權重、逐題結果與報告

In [9]:
import shutil
shutil.make_archive('/content/MMIP_Week7_results', 'zip', '/content/MMIP_Week7', 'artifacts')
files.download('/content/MMIP_Week7_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. 啟動聊天介面
執行後點開 Gradio 暫時連結。介面會在這個 Colab 執行階段運作；關閉執行階段後連結即失效。

In [ ]:
!python app.py --host 0.0.0.0 --share

`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1795075968.782763    7206 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795075971.699753    7206 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100% 2/2 [00:29<00:00, 14.71s/it]
/content/MMIP_Week7/ap

In [13]:
from pathlib import Path
import subprocess, sys, urllib.request
files = ('core.py', 'verify.py')
[Path(name).write_bytes(urllib.request.urlopen(f'https://raw.githubusercontent.com/Sharon0726/MMIP/main/Week7/{name}?v=8d6a763').read()) for name in files]
assert 'ERROR_CODE_RE' in Path('core.py').read_text(encoding='utf-8')
subprocess.run([sys.executable, 'verify.py'], check=True)

CompletedProcess(args=['/usr/bin/python3', 'verify.py'], returncode=0)